# Notebook 4 — EDA (Training Split Only)
All detailed exploration is performed on **training data only**. The test set is intentionally not opened, following the task instructions.

In [ ]:

import os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

ART = Path("../artifacts")
ART.mkdir(exist_ok=True)
CHARTS = ART / "charts"
CHARTS.mkdir(exist_ok=True)

import matplotlib.pyplot as plt

train = pd.read_csv(ART/"03_train.csv")
train["order_purchase_timestamp"] = pd.to_datetime(train["order_purchase_timestamp"], errors="coerce")
print("Shape:", train.shape)
print("Memory (MB):", round(train.memory_usage(deep=True).sum()/1024**2, 2))
display(train.dtypes.to_frame("dtype"))


In [ ]:

missing = train.isna().sum().sort_values(ascending=False)
missing = missing[missing>0].to_frame("missing")
missing["pct"] = missing["missing"]/len(train)
display(missing)


In [ ]:

num_cols = train.select_dtypes(include=np.number).columns.tolist()
display(train[num_cols].describe().T)


In [ ]:

cat_cols = train.select_dtypes(include=["object","category"]).columns.tolist()
cat_summary = []
for c in cat_cols:
    vc = train[c].value_counts(dropna=False)
    cat_summary.append({"column":c,"cardinality":train[c].nunique(dropna=True),"rare_1pct":int((vc/len(train)<0.01).sum())})
display(pd.DataFrame(cat_summary).sort_values("cardinality", ascending=False).head(20))


In [ ]:

# Label relationships
for c in ["customer_state","seller_state","order_status","order_purchase_timestamp"]:
    if c in train.columns and c != "order_purchase_timestamp":
        g = train.groupby(c)["late"].agg(["count","mean"]).sort_values("mean", ascending=False).head(15)
        display(g)


In [ ]:

# Date features and seasonality
train["purchase_weekday"] = train["order_purchase_timestamp"].dt.dayofweek
train["purchase_month"] = train["order_purchase_timestamp"].dt.month
weekday_rate = train.groupby("purchase_weekday")["late"].mean()
month_rate = train.groupby("purchase_month")["late"].mean()

plt.figure(figsize=(8,4))
weekday_rate.plot(kind="bar")
plt.title("Late-delivery rate by purchase weekday")
plt.ylabel("Late rate")
plt.tight_layout()
plt.savefig(CHARTS/"eda_late_by_weekday.png", dpi=150)
plt.show()

plt.figure(figsize=(8,4))
month_rate.plot(kind="bar")
plt.title("Late-delivery rate by purchase month")
plt.ylabel("Late rate")
plt.tight_layout()
plt.savefig(CHARTS/"eda_late_by_month.png", dpi=150)
plt.show()


In [ ]:

# Numerical relations to label
corr = train[num_cols + ["late"]].corr(numeric_only=True)["late"].sort_values(ascending=False)
display(corr.to_frame("correlation"))


In [ ]:

findings = f"""# EDA Findings Summary

- Training rows: {len(train):,}
- Target late rate: {train['late'].mean():.2%}
- Missing values were inspected column-by-column; missingness should be handled in the feature pipeline.
- Numerical distributions and categorical cardinality were reviewed to identify skew, outliers, and rare categories.
- Delivery risk was compared across categorical groups and calendar features.
- Date-derived features such as weekday/month are candidates because the task explicitly calls for date/seasonality analysis.
- The feature set should use only information available at prediction time; actual delivery timestamps and any post-delivery information must not be used as predictors.
"""
(ART/"04_eda_findings.md").write_text(findings, encoding="utf-8")
print(findings)
